# Construction Safety v1 — Exploratory Data Analysis

Dataset: Roboflow `svrd/construction-safety-gdvov` v1 (19 classes, YOLOv8 format).

Goals of this notebook:

1. Verify dataset health (image/label parity, valid class IDs).
2. Quantify class distribution per split — detect imbalance before training.
3. Distribution of annotations per image and image dimensions.
4. Co-occurrence of compliance vs. violation classes (Hardhat / NO-Hardhat, etc.).
5. Sample visualizations with bounding boxes.

All data is read through `datasets/{train,valid,test}/` symlinks generated by `scripts/prepare_dataset.py`.

In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from PIL import Image

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_YAML = PROJECT_ROOT / "config" / "data.yaml"
DATASETS = PROJECT_ROOT / "datasets"
SPLITS = ("train", "valid", "test")

with DATA_YAML.open() as f:
    cfg = yaml.safe_load(f)
CLASS_NAMES = cfg["names"]
NUM_CLASSES = cfg["nc"]
print(f"Loaded {NUM_CLASSES} classes from {DATA_YAML.relative_to(PROJECT_ROOT)}")
CLASS_NAMES

## 1. Sanity check: image/label parity + class ID validity

In [ ]:
def split_label_files(split: str) -> list[Path]:
    return sorted((DATASETS / split / "labels").glob("*.txt"))

def split_image_files(split: str) -> list[Path]:
    return sorted(p for p in (DATASETS / split / "images").iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})

rows = []
invalid_ids: list[tuple[str, str, int]] = []
for split in SPLITS:
    imgs = split_image_files(split)
    lbls = split_label_files(split)
    img_stems = {p.stem for p in imgs}
    lbl_stems = {p.stem for p in lbls}
    rows.append({
        "split": split,
        "images": len(imgs),
        "labels": len(lbls),
        "images_without_label": len(img_stems - lbl_stems),
        "labels_without_image": len(lbl_stems - img_stems),
    })
    for lbl in lbls:
        for line in lbl.read_text().splitlines():
            line = line.strip()
            if not line:
                continue
            cid = int(line.split()[0])
            if cid < 0 or cid >= NUM_CLASSES:
                invalid_ids.append((split, lbl.name, cid))

pd.DataFrame(rows).set_index("split")

In [ ]:
print(f"Invalid class IDs found: {len(invalid_ids)}")
if invalid_ids:
    print(invalid_ids[:10])

## 2. Class distribution per split

In [ ]:
def class_counts(split: str) -> Counter:
    c: Counter = Counter()
    for lbl in split_label_files(split):
        for line in lbl.read_text().splitlines():
            line = line.strip()
            if not line:
                continue
            c[int(line.split()[0])] += 1
    return c

counts_by_split = {split: class_counts(split) for split in SPLITS}
dist_df = pd.DataFrame(
    {split: [counts_by_split[split].get(i, 0) for i in range(NUM_CLASSES)] for split in SPLITS},
    index=CLASS_NAMES,
)
dist_df["total"] = dist_df.sum(axis=1)
dist_df = dist_df.sort_values("total", ascending=False)
dist_df

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
dist_df[list(SPLITS)].plot(kind="barh", stacked=True, ax=ax, color=["#3b82f6", "#f59e0b", "#10b981"])
ax.set_xlabel("Annotations")
ax.set_title("Class distribution by split (sorted by total)")
ax.invert_yaxis()
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Annotations per image

In [ ]:
ann_per_image: dict[str, list[int]] = {split: [] for split in SPLITS}
for split in SPLITS:
    for lbl in split_label_files(split):
        n = sum(1 for line in lbl.read_text().splitlines() if line.strip())
        ann_per_image[split].append(n)

summary = pd.DataFrame({
    "min": {s: min(ann_per_image[s]) for s in SPLITS},
    "max": {s: max(ann_per_image[s]) for s in SPLITS},
    "mean": {s: float(np.mean(ann_per_image[s])) for s in SPLITS},
    "median": {s: float(np.median(ann_per_image[s])) for s in SPLITS},
    "empty": {s: sum(1 for x in ann_per_image[s] if x == 0) for s in SPLITS},
})
summary

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist([ann_per_image[s] for s in SPLITS], bins=range(0, max(max(v) for v in ann_per_image.values()) + 2),
        label=list(SPLITS), color=["#3b82f6", "#f59e0b", "#10b981"], stacked=True)
ax.set_xlabel("Annotations per image")
ax.set_ylabel("Images")
ax.set_title("Distribution of annotations per image")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Image dimensions

In [ ]:
dim_rows = []
for split in SPLITS:
    for img in split_image_files(split):
        with Image.open(img) as im:
            w, h = im.size
        dim_rows.append({"split": split, "width": w, "height": h})
dim_df = pd.DataFrame(dim_rows)
dim_df.groupby("split")[["width", "height"]].describe()[[("width", "min"), ("width", "max"), ("height", "min"), ("height", "max")]]

In [ ]:
size_counts = dim_df.groupby(["width", "height"]).size().reset_index(name="n").sort_values("n", ascending=False).head(10)
size_counts

## 5. Co-occurrence: compliance vs. violation classes per image

How often does the same image contain `Hardhat` and `NO-Hardhat` (i.e., mixed compliance scenes)? Same for Mask and Safety Vest.

In [ ]:
PAIRS = [("Hardhat", "NO-Hardhat"), ("Mask", "NO-Mask"), ("Safety Vest", "NO-Safety Vest")]
name_to_id = {n: i for i, n in enumerate(CLASS_NAMES)}

co_rows = []
for split in SPLITS:
    for lbl in split_label_files(split):
        ids = {int(line.split()[0]) for line in lbl.read_text().splitlines() if line.strip()}
        for pos, neg in PAIRS:
            p_id, n_id = name_to_id[pos], name_to_id[neg]
            co_rows.append({
                "split": split,
                "pair": f"{pos} / {neg}",
                "only_positive": int(p_id in ids and n_id not in ids),
                "only_negative": int(n_id in ids and p_id not in ids),
                "both": int(p_id in ids and n_id in ids),
                "neither": int(p_id not in ids and n_id not in ids),
            })
co_df = pd.DataFrame(co_rows).groupby(["split", "pair"]).sum()
co_df

## 6. Sample visualizations with bounding boxes

In [ ]:
import random
from matplotlib.patches import Rectangle

random.seed(42)

def yolo_boxes(label_path: Path, w: int, h: int) -> list[tuple[int, float, float, float, float]]:
    out = []
    for line in label_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        cid, xc, yc, bw, bh = int(parts[0]), *map(float, parts[1:])
        x = (xc - bw / 2) * w
        y = (yc - bh / 2) * h
        out.append((cid, x, y, bw * w, bh * h))
    return out

COLORS = plt.cm.tab20(np.linspace(0, 1, NUM_CLASSES))

sample_imgs = random.sample(split_image_files("train"), 6)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, img_path in zip(axes.flat, sample_imgs):
    with Image.open(img_path) as im:
        w, h = im.size
        ax.imshow(im)
    lbl_path = DATASETS / "train" / "labels" / f"{img_path.stem}.txt"
    if lbl_path.exists():
        for cid, x, y, bw, bh in yolo_boxes(lbl_path, w, h):
            ax.add_patch(Rectangle((x, y), bw, bh, fill=False, edgecolor=COLORS[cid], linewidth=2))
            ax.text(x, y - 3, CLASS_NAMES[cid], color="white",
                    bbox=dict(facecolor=COLORS[cid], alpha=0.7, pad=1, edgecolor="none"),
                    fontsize=8)
    ax.set_title(img_path.name[:30], fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Findings (filled in after running cells)

Expected highlights (precomputed offline):

- **Image/label parity**: 521/114/82, no missing files.
- **Most frequent classes (by total annotations)**: `Person` (1148), `Safety Cone` (600), `NO-Safety Vest` (582), `Hardhat` (574), `NO-Mask` (491), `Safety Vest` (424), `NO-Hardhat` (402).
- **Sparse classes (≤60 annotations total)**: `trailer` (15), `van` (28), `truck` (35), `machinery` (45), `sedan` (54), `Ladder` (58). The model will struggle on these — consider class weighting or focal loss in later phases.
- **Violation classes (`NO-*`) are well represented** — direct supervised signal for compliance detection, no need to infer absence in v1.
- **Vehicle taxonomy is messy**: `vehicle` (122) overlaps with `sedan / van / truck / dump truck / wheel loader / trailer`. Consider collapsing to a coarser taxonomy for v2 unless downstream alerts need the fine-grained types.